In [1]:
# 82. マスクの top-10 予測と確率計算

from transformers import BertTokenizer, BertForMaskedLM
import torch
import torch.nn.functional as F

def top_k_mask_predictions(text, k=10):
    tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
    model = BertForMaskedLM.from_pretrained('bert-base-uncased')
    model.eval()

    # トークナイズ・ID化
    inputs = tokenizer.encode_plus(text, return_tensors='pt')
    input_ids = inputs['input_ids']
    mask_idx = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]

    # 推論
    with torch.no_grad():
        logits = model(**inputs).logits

    # マスク位置のロジット取得
    mask_logits = logits[0, mask_idx, :]
    probs = F.softmax(mask_logits, dim=-1)

    # 上位 k
    topk_probs, topk_indices = torch.topk(probs, k, dim=-1)
    for prob, idx in zip(topk_probs[0], topk_indices[0]):
        token = tokenizer.convert_ids_to_tokens(int(idx))
        print(f"{token}\t{prob.item():.4f}")

if __name__ == "__main__":
    text = "The movie was full of [MASK]."
    top_k_mask_predictions(text, k=10)


c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\eriya\anaconda3\envs\python-env\Lib\site-packages\sklearn\utils\_param_validation.py:11: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  from scipy.sparse import csr_matrix, issparse
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected

fun	0.1071
surprises	0.0663
drama	0.0447
stars	0.0272
laughs	0.0254
action	0.0195
excitement	0.0190
people	0.0183
tension	0.0150
music	0.0146
